<a href="https://colab.research.google.com/github/somendrew/LMS_Image_Classification/blob/main/LMS_Offical_Image_Stealth_Scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Warnings Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

### Library Installation

In [2]:
!pip install -q torch torchvision transformers pillow

### Import Pandas

In [3]:
import pandas as pd

### Google Colab Authentication and Gspread Setup

In [4]:
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Get credentials and authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

### Commented Out Google Sheet Opening

In [5]:
# # Open the sheet by its name
# spreadsheet = gc.open('  P0.1 Sports {Exploratory analysis} || FIFA || Somendra ')

# # Select the first worksheet (tab)
# worksheet = spreadsheet.sheet1

In [6]:
spreadsheet = gc.open('test_set_LMS')

worksheet = spreadsheet.worksheet('Sheet6')

### Open Specific Google Sheet and Worksheet

### Display Worksheet Object

In [7]:
worksheet

<Worksheet 'Sheet6' id:1553180624>

### Load Data into DataFrame

In [8]:
# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to a DataFrame (using the first row as headers)
df = pd.DataFrame(rows[1:], columns=rows[0])

# View the first few rows
df.head()

,MID,MID,Athlete Name,Getty Image link,Image 1,Getty Image link,Image 2,Getty Image link,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5
0,/g/11bwf0rx9b,/g/11bwf0rx9b,Jurij Medveděv,,,,,,,,,,
1,/g/11fmsrdvgv,/g/11fmsrdvgv,Ricardo Rodriguez,,,,,,,,,,
2,/g/11mxmg71n5,/g/11mxmg71n5,Lucas Villalba,,,,,,,,,,
3,/g/11fcq4l7r6,/g/11fcq4l7r6,Xander Blomme,,,,,,,,,,
4,/g/11gphlh9t2,/g/11gphlh9t2,Yousef Abu Al-Jazar,,,,,,,,,,


### Create DataFrame Copy and Check Shape

In [9]:
#Taking sample df

new_df = df[:]
new_df.shape

(43, 13)

### Display Head of Copied DataFrame

In [10]:
new_df.head()

,MID,MID,Athlete Name,Getty Image link,Image 1,Getty Image link,Image 2,Getty Image link,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5
0,/g/11bwf0rx9b,/g/11bwf0rx9b,Jurij Medveděv,,,,,,,,,,
1,/g/11fmsrdvgv,/g/11fmsrdvgv,Ricardo Rodriguez,,,,,,,,,,
2,/g/11mxmg71n5,/g/11mxmg71n5,Lucas Villalba,,,,,,,,,,
3,/g/11fcq4l7r6,/g/11fcq4l7r6,Xander Blomme,,,,,,,,,,
4,/g/11gphlh9t2,/g/11gphlh9t2,Yousef Abu Al-Jazar,,,,,,,,,,


## Image Processor

### Image Processing Setup and Functions

This section initializes pre-trained deep learning models for object detection (DETR for counting people) and visual question answering (BLIP for asking yes/no questions about images). It also defines helper functions `count_people` and `ask_blip`, along with a main `analyze_profile_photo` function to apply these models for validating image content.

In [ ]:
import torch
from PIL import Image
from transformers import (
    DetrImageProcessor, DetrForObjectDetection,
    BlipProcessor, BlipForQuestionAnswering
)
from io import BytesIO # Added this import

# ─── Load models once ──────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"

# Person counter — DETR
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device)
detr_model.eval()

# Visual Q&A — BLIP
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)
blip_model.eval()

# ─── Helper functions ──────────────────────────────────────────

def count_people(image: Image.Image, threshold: float = 0.85) -> int:
    """Returns number of people detected in the image."""
    inputs = detr_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = detr_model(**inputs)

    results = detr_processor.post_process_object_detection(
        outputs,
        target_sizes=torch.tensor([image.size[::-1]]),
        threshold=threshold
    )[0]

    labels = results["labels"].tolist()
    # COCO label 1 = person
    return labels.count(1)


def ask_blip(image: Image.Image, question: str) -> bool:
    """Ask a yes/no question about the image using BLIP VQA."""
    inputs = blip_processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        output = blip_model.generate(**inputs)
    answer = blip_processor.decode(output[0], skip_special_tokens=True).strip().lower()
    return answer.startswith("yes")


# ─── Main analyzer ──────────────────────────────────────────
def analyze_profile_photo(image_source: str) -> dict:
    if image_source.startswith("http://") or image_source.startswith("https://"):
        response = requests.get(image_source)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_source).convert("RGB")

    person_count = count_people(image)
    exactly_one_person = (person_count == 1)

    front_facing = ask_blip(
        image,
        "Is the person's face directly facing the camera?"
    )
    consistent_background = ask_blip(
        image,
        "Is the background plain, uniform ?"
    )
    official_style = ask_blip(
        image,
        "Does this look like a professional ID photo with good lighting and sharp focus?"
    )
    wearing_jersey = ask_blip(
        image,
        "Is the person in the photo, wearning sports jersey?"
    )

    passes = all([exactly_one_person, front_facing, consistent_background, official_style,wearing_jersey])

    return {
        "exactly_one_person": exactly_one_person,
        "person_count": person_count,
        "front_facing": front_facing,
        "consistent_background": consistent_background,
        "official_style": official_style,
        "passes": passes,
    }

## Iterate through DataFrame and fill the links

### Cleaning the entity name


### Simplify Name Function

This helper function removes diacritics (accents) from text, normalizing names for consistent processing and querying, especially useful for web scraping queries.

In [12]:

import unicodedata

def simplify_name(text):
    # Normalize the unicode to decomposition form (NFD)
    # This separates "ā" into "a" + "◌̄"
    normalized = unicodedata.normalize('NFD', text)

    # Filter out the non-spacing mark characters (the accents)
    simplified = "".join(c for c in normalized if unicodedata.category(c) != 'Mn')

    return simplified

### Web Scraping and Image Filtering Loop

This loop iterates through the DataFrame, scrapes images from Getty Images for each athlete, filters them based on keywords, and then applies the `analyze_profile_photo` function to select suitable profile pictures. The links to these validated images are then populated back into the DataFrame.

In [13]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import time
player_images = []

def stealth_scrape(entity):
    url = f"https://www.gettyimages.com/photos/{entity.replace(' ', '-')}"

    response = requests.get(url, impersonate="chrome120")

    images_found = []  # Local list instead of global

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        images = soup.find_all('img')
        for img in images[:20]:
            src = img.get('src')
            if src:  # Also guard against None src values
                images_found.append(src)
    else:
        print(f"Blocked! Status Code: {response.status_code}")

    return images_found  # Always return a list (empty if blocked)


# Iterate through the DataFrame
print("Starting bulk scrape...")

for index, row in new_df.iterrows():

    uncleaned_entity = row['Athlete Name']
    entity = simplify_name(uncleaned_entity)

    query = f'{entity} Portrait'

    print(f"Processing: {entity}")

    # Fetch links
    player_images = stealth_scrape(query)

    # Splitting entity to check for individual keywords (case-insensitive) present in the link
    keywords = [k.lower() for k in entity.split()]

    #Defining number of getty columns we have in sheet
    max_columns = 5

    filtered_images = [ link for link in player_images if all(word in link.lower() for word in keywords)]
    print(f"Total images found: {len(filtered_images)}")

    vllm_images = [img for img in filtered_images if analyze_profile_photo(img)["passes"]] #true if passes all test cases
    print(f"Total images after vllm_processing: {len(vllm_images)}")

    # Images we need (5)
    images_to_save = vllm_images[:max_columns]

    # Fill the dataframe columns
    for i, link in enumerate(images_to_save):
        new_df.at[index, f'Getty Image link {i+1}'] = link

    # Respectful delay to avoid IP bans
    time.sleep(2)



Starting bulk scrape...
Processing: Jurij Medvedev
Total images found: 6
Total images after vllm_processing: 6
Processing: Ricardo Rodriguez
Total images found: 12
Total images after vllm_processing: 8
Processing: Lucas Villalba
Total images found: 0
Total images after vllm_processing: 0
Processing: Xander Blomme
Total images found: 0
Total images after vllm_processing: 0
Processing: Yousef Abu Al-Jazar
Total images found: 0
Total images after vllm_processing: 0
Processing: Ahmed Eid
Total images found: 5
Total images after vllm_processing: 0
Processing: Abdulrahman Al-Sanbi
Total images found: 2
Total images after vllm_processing: 0
Processing: Mathis Suray
Total images found: 3
Total images after vllm_processing: 0
Processing: Oh Hyeon-gyu
Total images found: 10
Total images after vllm_processing: 0
Processing: Adamu Alhassan
Total images found: 0
Total images after vllm_processing: 0
Processing: Hugo Vogel
Total images found: 6
Total images after vllm_processing: 0
Processing: Phili

### Display Updated DataFrame

In [14]:
new_df

,MID,MID,Athlete Name,Getty Image link,Image 1,Getty Image link,Image 2,Getty Image link,Image 3,Getty Image link 4,Image 4,Getty Image link 5,Image 5,Getty Image link 1,Getty Image link 2,Getty Image link 3
0,/g/11bwf0rx9b,/g/11bwf0rx9b,Jurij Medveděv,,,,,,,https://media.gettyimages.com/id/2171695539/ph...,,https://media.gettyimages.com/id/2171695532/ph...,,https://media.gettyimages.com/id/2171697745/ph...,https://media.gettyimages.com/id/2171695548/ph...,https://media.gettyimages.com/id/2171695541/ph...
1,/g/11fmsrdvgv,/g/11fmsrdvgv,Ricardo Rodriguez,,,,,,,https://media.gettyimages.com/id/2157305966/ph...,,https://media.gettyimages.com/id/1153841479/ph...,,https://media.gettyimages.com/id/2157305988/ph...,https://media.gettyimages.com/id/2157169935/ph...,https://media.gettyimages.com/id/2157305968/ph...
2,/g/11mxmg71n5,/g/11mxmg71n5,Lucas Villalba,,,,,,,,,,,NaN,NaN,NaN
3,/g/11fcq4l7r6,/g/11fcq4l7r6,Xander Blomme,,,,,,,,,,,NaN,NaN,NaN
4,/g/11gphlh9t2,/g/11gphlh9t2,Yousef Abu Al-Jazar,,,,,,,,,,,NaN,NaN,NaN
5,/g/11j4qp2pwb,/g/11j4qp2pwb,Ahmed Eid,,,,,,,,,,,NaN,NaN,NaN
6,/g/11lx8y2y06,/g/11lx8y2y06,Abdulrahman Al-Sanbi,,,,,,,,,,,NaN,NaN,NaN
7,/g/11f15n5l8c,/g/11f15n5l8c,Mathis Suray,,,,,,,,,,,NaN,NaN,NaN
8,/g/11h18l1315,/g/11h18l1315,Oh Hyeon-gyu,,,,,,,,,,,NaN,NaN,NaN
9,/g/11h3dnrb71,/g/11h3dnrb71,Adamu Alhassan,,,,,,,,,,,NaN,NaN,NaN


## Direct Update

### Update Google Sheet with DataFrame

In [17]:
import numpy as np

# 1. Convert DataFrame back to a list of lists (including headers)
# Fill NaN values with empty strings before converting to list of lists
new_df_cleaned = new_df.replace({np.nan: ''})
data_to_upload = [new_df_cleaned.columns.tolist()] + new_df_cleaned.values.tolist()

# 2. Update the sheet starting from cell A1
worksheet.update('A1', data_to_upload)

{'spreadsheetId': '1ZxpzmUjGdR_iyWA_qpIjNCgj-M9oaPn5v72AaCpJY6Q',
 'updatedRange': 'Sheet6!A1:P44',
 'updatedRows': 44,
 'updatedColumns': 16,
 'updatedCells': 704}